# RFd3 Generation Pipeline
This notebook implements the interactive single-file generation pipeline using the Foundry environment for generating protein-protein binding loops via **RoseTTAFold Diffusion 3 (RFd3)**, sequences via **LigandMPNN**, and structural validation via **RoseTTAFold 3 (RF3)**.

## Setup Environment

In [1]:
import os, time, sys
import dataclasses
import re
import pandas as pd
import numpy as np
import torch
from biotite.structure.io.pdbx import CIFFile, get_structure as get_cif_structure
from biotite.structure.io.pdb import PDBFile
from biotite.structure import superimpose, rmsd
from biotite.sequence import ProteinSequence

from atomworks.io.utils.io_utils import to_cif_file
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES

# Foundry Inference imports
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput
from rfd3.inference.input_parsing import DesignInputSpecification

torch.set_float32_matmul_precision('medium')
import logging
logging.getLogger("transforms").setLevel(logging.ERROR)
logging.getLogger("atomworks.io").setLevel(logging.ERROR)
logging.getLogger("atomworks.ml").setLevel(logging.ERROR)
logging.getLogger("foundry").setLevel(logging.ERROR)

def get_sequence_from_array(atom_array, chain_id="A"):
    mask = (atom_array.chain_id == chain_id) & (atom_array.atom_name == "CA")
    ca_atoms = atom_array[mask]
    if len(ca_atoms) == 0: return ""
    
    sort_idx = np.argsort(ca_atoms.res_id)
    res_names = ca_atoms.res_name[sort_idx]
    
    seq_letters = []
    for rn in res_names:
        try:
            seq_letters.append(ProteinSequence.convert_letter_3to1(rn))
        except Exception:
            seq_letters.append("X")
    return "".join(seq_letters)

def get_pdb_length(pdb_path, chain_id):
    try:
        pdb_file = PDBFile.read(pdb_path)
        atom_array = pdb_file.get_structure()[0]
        return len(get_sequence_from_array(atom_array, chain_id))
    except Exception:
        count = 0
        with open(pdb_path, 'r') as f:
            for line in f:
                if line.startswith('ATOM') and line.split()[4] == chain_id and line.split()[2] == "CA":
                    count += 1
        return count

def renumber_atom_array_residues(atom_array):
    new_res_ids = np.zeros(len(atom_array), dtype=int)
    for chain_id in np.unique(atom_array.chain_id):
        chain_mask = atom_array.chain_id == chain_id
        chain_res_ids = atom_array.res_id[chain_mask]
        
        unique_old_ids = []
        last_id = None
        for r_id in chain_res_ids:
            if r_id != last_id:
                unique_old_ids.append(r_id)
                last_id = r_id
                
        id_map = {old_id: new_id for new_id, old_id in enumerate(unique_old_ids, start=1)}
        new_res_ids[chain_mask] = [id_map[old_id] for old_id in chain_res_ids]
        
    atom_array.res_id = new_res_ids
    return atom_array

#seed_everything(42)
checkpoint_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "Tools", "foundry_checkpoints"))
os.environ["FOUNDRY_CHECKPOINT_DIRS"] = checkpoint_dir
os.environ["DGLBACKEND"] = "pytorch"

Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Failed to get GPU information from pynvml: Not Supported. Using default values for NVIDIA GeForce RTX 5070 Ti
NVIDIA GeForce GTX 1660 Ti.


GPU information: NVIDIA RTX A6000, 8, 6, 45, 84, 300, 2100


14:30:44 DEBUG transforms: Debug mode is on
14:30:45 INFO rdkit: Enabling RDKit 2025.03.6 jupyter extensions


## Generate Contigs and Length Constraints

In [2]:
pdb_complex_file_name = "TIMP3_vs_ADAM17_HADDOCK_Xray.pdb"
data_path = "../Data"
pdb_path = os.path.join(data_path, pdb_complex_file_name)

output_prefix = "design"
num_sequences_to_generate = 5

loop_names = ["C", "EF"]
chain_to_design = "A"
fixed_chains = ["B"]
total_length = 121  # Max native length of targeted chain A

loop_configs = {
    "AB": {"normal": 6, "max": 15, "pos": 30, "left": "LVK", "right": "LVY"},
    "C": {"normal": 6, "max": 15, "pos": 62, "left": "HTE", "right": "GLK"},
    "EF": {"normal": 4, "max": 10, "pos": 92, "left": "MYT", "right": "FVE"},
    "GH": {"normal": 10, "max": 20, "pos": 127, "left": "KSC", "right": "NEC"},
    "Multi": {"normal": 10, "max": 20, "pos": 143, "left": "LWT", "right": "YQS"}
}

selected_loops = [loop_configs[name] for name in loop_names]
selected_loops.sort(key=lambda x: x["pos"])

contig_parts = []
current_pos = 1
for loop in selected_loops:
    if current_pos <= loop["pos"]:
        contig_parts.append(f"{chain_to_design}{current_pos}-{loop['pos']}")
    contig_parts.append(f"{loop['normal']}-{loop['max']}")
    current_pos = loop["pos"] + loop["normal"] + 1

if current_pos <= total_length:
    contig_parts.append(f"{chain_to_design}{current_pos}-{total_length}")

contig_string = ",".join(contig_parts)

fix_chain_len = get_pdb_length(pdb_path, fixed_chains[0])
full_contig_string = f"{contig_string},/0,{fixed_chains[0]}1-{fix_chain_len}"
print(f"Final Contig String: {full_contig_string}")

min_inpaint_len = sum(loop["normal"] for loop in selected_loops)
max_inpaint_len = sum(loop["max"] for loop in selected_loops)
native_removed_len = sum(loop["normal"] for loop in selected_loops)

base_length = total_length + fix_chain_len - native_removed_len
total_output_length_min = base_length + min_inpaint_len
total_output_length_max = base_length + max_inpaint_len
print(f"Total length bounds: {total_output_length_min}-{total_output_length_max}")

Final Contig String: A1-62,6-15,A69-92,4-10,A97-121,/0,B1-255
Total length bounds: 376-391


## Run RFd3 Generation

In [3]:
out_name = pdb_complex_file_name.replace(".pdb", "").replace(".cif", "")
rfd3_out_dir = f"../Local/rfd3_output/{out_name}"
lmpnn_out_dir = f"../Local/ligandmpnn_output/{out_name}"
rf3_out_dir = f"../Local/rf3_output/{out_name}"
os.makedirs(rfd3_out_dir, exist_ok=True)
os.makedirs(lmpnn_out_dir, exist_ok=True)
os.makedirs(rf3_out_dir, exist_ok=True)

print("\n--- Running RFd3 ---")
rfd3_config = RFD3InferenceConfig(
    diffusion_batch_size=min(10, num_sequences_to_generate),
    low_memory_mode=False,
    specification={'length': f"{total_output_length_min}-{total_output_length_max}", 'contig': full_contig_string, 'extra': {}}
)
rfd3_engine = RFD3InferenceEngine(**dataclasses.asdict(rfd3_config))

spec_input = DesignInputSpecification(
    input=pdb_path,
    contig=full_contig_string, 
    length=f"{total_output_length_min}-{total_output_length_max}",
    extra={}
)

batches_needed = num_sequences_to_generate // rfd3_config.diffusion_batch_size + (1 if num_sequences_to_generate % rfd3_config.diffusion_batch_size != 0 else 0)

rfd3_outputs_dict = rfd3_engine.run(inputs=spec_input, n_batches=batches_needed, out_dir=None)

generated_arrays = []
if rfd3_outputs_dict:
    for key, rfd3_out_list in rfd3_outputs_dict.items():
        if not key.startswith("backbone"): continue
        for batch_idx, rfd3_out in enumerate(rfd3_out_list):
            if batch_idx >= num_sequences_to_generate: break
            design_id = f"{output_prefix}_{batch_idx}"
            
            clean_array = renumber_atom_array_residues(rfd3_out.atom_array)
            to_cif_file(clean_array, f"{rfd3_out_dir}/{design_id}.cif", file_type="cif")
            generated_arrays.append((design_id, clean_array))

del rfd3_engine
torch.cuda.empty_cache()
print(f"Saved {len(generated_arrays)} backbone designs.")


--- Running RFd3 ---


Using bfloat16 Automatic Mixed Precision (AMP)
14:32:02 INFO rfd3.engine: [rank: 0] Finished inference batch in 49.61 seconds.


Saved 5 backbone designs.


## LigandMPNN Sequence Inpainting

In [4]:
print(f"\n--- LigandMPNN ---")
lmpnn_engine = MPNNInferenceEngine(model_type="ligand_mpnn", is_legacy_weights=True, write_structures=False, write_fasta=True, out_directory=lmpnn_out_dir)
final_records = []

for idx, (design_id, rfd3_array) in enumerate(generated_arrays):
    aa_sequence = get_sequence_from_array(rfd3_array, chain_to_design)
    
    fixed_positions_A = []
    current_fixed_start = 1
    current_seq_idx = 0
    
    for loop in selected_loops:
        flank_left = loop["left"]
        flank_right = loop["right"]
        regex_pattern = re.compile(f"{flank_left}([A-Z]+?){flank_right}")
        match = regex_pattern.search(aa_sequence[current_seq_idx:])
        if match:
            match_start = current_seq_idx + match.start()
            match_end = current_seq_idx + match.end()
            inserted_seq = match.group(1)
            loop_start_1idx = match_start + len(flank_left) + 1
            
            fixed_positions_A.extend(range(current_fixed_start, loop_start_1idx))
            current_fixed_start = loop_start_1idx + len(inserted_seq)
            current_seq_idx = match_end - len(flank_right)
    
    fixed_positions_A.extend(range(current_fixed_start, len(aa_sequence) + 1))
    fixed_residues_str = [f"{chain_to_design}{pos}" for pos in fixed_positions_A]
    
    b_chain_mask = (rfd3_array.chain_id == fixed_chains[0]) & (rfd3_array.atom_name == "CA")
    b_res_ids = np.unique(rfd3_array.res_id[b_chain_mask])
    fixed_residues_str.extend([f"{fixed_chains[0]}{pos}" for pos in b_res_ids])

    mpnn_input_dict = {
        "name": design_id,
        "batch_size": 1,
        "remove_waters": True,
        "seed": 42,
        "fixed_residues": fixed_residues_str,
        "sampling_temp": 0.1
    }
    
    mpnn_outputs = lmpnn_engine.run(input_dicts=[mpnn_input_dict], atom_arrays=[rfd3_array])
    
    for seq_idx, mpnn_out in enumerate(mpnn_outputs):
        valid_mask = ~np.isnan(mpnn_out.atom_array.coord[:, 0])
        lmpnn_array = mpnn_out.atom_array[valid_mask]
        lmpnn_array = renumber_atom_array_residues(lmpnn_array)
        
        full_seq_designed = get_sequence_from_array(lmpnn_array, chain_to_design)
        to_cif_file(lmpnn_array, f"{lmpnn_out_dir}/{design_id}_mpnn{seq_idx}.cif", file_type="cif")
        
        loop_data = {}
        curr_idx = 0
        for name_idx, loop in enumerate(selected_loops):
            loop_name = loop_names[name_idx]
            f_left = loop["left"]
            f_right = loop["right"]
            m = re.search(f"{f_left}(.*?){f_right}", full_seq_designed[curr_idx:])
            
            if m:
                seq = m.group(1)
                loop_data[f"loop_{loop_name}_seq"] = seq
                loop_data[f"loop_{loop_name}_length"] = len(seq)
                curr_idx += m.end() - len(f_right)
            else:
                loop_data[f"loop_{loop_name}_seq"] = "MISSING"
                loop_data[f"loop_{loop_name}_length"] = 0
        
        final_records.append({
            "design_id": design_id,
            "seq_idx": seq_idx,
            **loop_data,
            "seq_recovery": float(getattr(mpnn_out, "output_dict", {}).get("sequence_recovery", 0.0)),
            "full_seq": full_seq_designed,
            "lmpnn_array": lmpnn_array,
            "rfd3_array": rfd3_array
        })

del lmpnn_engine
torch.cuda.empty_cache()
print(f"Generated sequences for {len(final_records)} designs.")


--- LigandMPNN ---
Generated sequences for 5 designs.


## RF3 Forward Folding & Validation

In [5]:
print(f"\n--- Running RF3 Validations ---")
rf3_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)
metric_records = []

for job in final_records:
    design_id = f"{job['design_id']}_mpnn{job['seq_idx']}"
    lmpnn_array = job['lmpnn_array']
    rfd3_array = job['rfd3_array']
                
    valid_atoms = ['N', 'CA', 'C', 'O', 'CB']
    array_for_rf3 = lmpnn_array[np.isin(lmpnn_array.atom_name, valid_atoms)].copy()
                
    input_structure = InferenceInput.from_atom_array(array_for_rf3, example_id=design_id)
    rf3_outputs_dict = rf3_engine.run(inputs=input_structure)
                
    rf3_target_key = next((k for k in rf3_outputs_dict.keys() if design_id in k), list(rf3_outputs_dict.keys())[0] if rf3_outputs_dict else None)

    if rf3_target_key and rf3_target_key in rf3_outputs_dict:
        rf3_output = rf3_outputs_dict[rf3_target_key][0]
        rf3_atom_array = rf3_output.atom_array
                    
        rf3_atom_array = renumber_atom_array_residues(rf3_atom_array)
        to_cif_file(rf3_atom_array, f"{rf3_out_dir}/{design_id}_refolded.cif", file_type="cif")

        bb_mask_rfd3 = np.isin(rfd3_array.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
        bb_mask_rf3 = np.isin(rf3_atom_array.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
                    
        bb_generated = rfd3_array[bb_mask_rfd3]
        bb_refolded = rf3_atom_array[bb_mask_rf3]
        bb_generated = bb_generated[bb_generated.atom_name != "OXT"]
        bb_refolded = bb_refolded[bb_refolded.atom_name != "OXT"]
                    
        if len(bb_generated) != len(bb_refolded):
            min_len = min(len(bb_generated), len(bb_refolded))
            bb_generated = bb_generated[:min_len]
            bb_refolded = bb_refolded[:min_len]

        bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
        overall_rmsd = rmsd(bb_generated, bb_refolded_fitted)
                    
        summary = rf3_output.summary_confidences
        plddt = summary.get('overall_plddt', 0.0)
        ptm = summary.get('ptm', 0.0)
                    
        loop_kwargs = {k: v for k, v in job.items() if k.startswith("loop_")}
                    
        metric_records.append({
            "file": f"{design_id}.cif",
            **loop_kwargs,
            "overall_rmsd": overall_rmsd,
            "plddt": plddt,
            "ptm": ptm,
            "seq_recovery": job["seq_recovery"],
            "full_seq": job["full_seq"]
        })

del rf3_engine
torch.cuda.empty_cache()
print("Validation complete.")


--- Running RF3 Validations ---


14:32:52 INFO rf3.inference_engines.rf3: [rank: 0] Loading checkpoint from /home/ryangustafson/Documents/GitHubProj/PhD-Research/Tools/foundry_checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
Using bfloat16 Automatic Mixed Precision (AMP)
14:33:02 WARNING rf3.inference_engines.rf3: [rank: 0] out_dir is None - results will be returned in memory! If you want to save to disk, please provide an out_dir.
14:33:02 INFO rf3.inference_engines.rf3: [rank: 0] Found 1 structures to predict!
14:33:02 INFO rf3.inference_engines.rf3: [rank: 0] Predicting structure 1/1: design_0_mpnn0
14:33:33 WARNING rf3.inference_engines.rf3: [rank: 0] out_dir is None - results will be returned in memory! If you want to save to disk, please provide an out_dir.
14:33:33 INFO rf3.inference_engines.rf3: [rank: 0] Found 1 structures to predict!
14:33:33 INFO rf3.inference_engines.rf3: [rank: 0] Predicting structure 1/1: design_1_mpnn0
14:34:03 WARNING rf3.inference_engines.rf3: [rank: 0] out_dir is None - results 

Validation complete.


## Summarize Metrics

In [7]:
if metric_records:
    df = pd.DataFrame(metric_records)
    df.to_csv(os.path.join(lmpnn_out_dir, "design_summary.csv"), index=False)
            
    seq_cols = [c for c in df.columns if c.startswith("loop_") and c.endswith("_seq")]
    len_cols = [c for c in df.columns if c.startswith("loop_") and c.endswith("_length")]
            
    clean_df = df.dropna(subset=seq_cols + ["plddt"]).copy()
    if not clean_df.empty:
        best_per_loop = clean_df.sort_values("plddt", ascending=False).groupby(seq_cols, as_index=False).first()
                
        avg_stats = clean_df.groupby(len_cols, as_index=False).agg(
            avg_plddt=("plddt", "mean"),
            avg_rmsd=("overall_rmsd", "mean"),
            count=("plddt", "count")
        ).sort_values(len_cols)
                
        best_per_length = best_per_loop.sort_values(len_cols + ["plddt"], ascending=[True]*len(len_cols) + [False]).groupby(len_cols, group_keys=False).head(5)
        best_per_length = best_per_length.merge(avg_stats, on=len_cols, how="left")
        best_per_length = best_per_length.sort_values(len_cols + ["plddt"], ascending=[True]*len(len_cols) + [False])
                
        best_per_length.to_csv(os.path.join(lmpnn_out_dir, "best_loops_per_length.csv"), index=False)
        avg_stats.to_csv(os.path.join(lmpnn_out_dir, "loop_length_averages.csv"), index=False)
        
        print(f"Saved summaries to {lmpnn_out_dir}")
else:
    print("No valid metric records generated.")

Saved summaries to ../Local/ligandmpnn_output/TIMP3_vs_ADAM17_HADDOCK_Xray


## Data Visualization

In [10]:
import py3Dmol
import pandas as pd
import os

# Helper to dynamically find the B-factor boundaries in the CIF
def get_b_metrics(cif_path):
    if not os.path.exists(cif_path):
        return 0.0, 1.0, False
        
    with open(cif_path, "r") as f:
        lines = f.readlines()
        
    col_b, col_atom = -1, -1
    col_count = 0
    b_factors = []
    header_section = True
    
    for line in lines:
        line = line.strip()
        if not line: continue
        if line.startswith('_atom_site.'):
            if '_atom_site.B_iso_or_equiv' in line: col_b = col_count
            if '_atom_site.label_atom_id' in line or '_atom_site.auth_atom_id' in line: col_atom = col_count
            col_count += 1
        elif line.startswith('ATOM') or line.startswith('HETATM'):
            header_section = False
            parts = line.split()
            if col_b != -1 and col_atom != -1 and len(parts) > max(col_b, col_atom):
                atom_name = parts[col_atom].strip('"').strip("'")
                if atom_name == "CA":
                    try:
                        b_factors.append(float(parts[col_b]))
                    except ValueError: continue
        elif not header_section and line.startswith('#'):
            break
            
    if not b_factors: return 0.0, 1.0, False
    
    b_min, b_max = min(b_factors), max(b_factors)
    avg_b = sum(b_factors) / len(b_factors)
    
    if b_max <= 1.1: invert = avg_b < 0.3
    else: invert = avg_b < 30
        
    return b_min, b_max, invert

# Calculate a 0-1 combined metric stringing together pLDDT, pTM, and RMSD
def calculate_combined_score(row):
    p_score = row['plddt'] / 100.0 if row['plddt'] > 1.0 else row['plddt']
    ptm_score = row.get('ptm', 0.0)
    # RMSD Factor (Decays to 0 at >= 4 Angstroms)
    rmsd_score = max(0.0, 1.0 - (row['overall_rmsd'] / 4.0))
    
    # 40% pLDDT, 40% pTM, 20% RMSD
    total = (p_score * 0.4) + (ptm_score * 0.4) + (rmsd_score * 0.2)
    return round(total, 3)

if os.path.exists(os.path.join(lmpnn_out_dir, "best_loops_per_length.csv")):
    best_df = pd.read_csv(os.path.join(lmpnn_out_dir, "best_loops_per_length.csv"))
    
    # Apply heuristic and sort by the new combined binding probability
    best_df['binding_probability'] = best_df.apply(calculate_combined_score, axis=1)
    top_design = best_df.sort_values("binding_probability", ascending=False).iloc[0]
    
    print("--- Top Design Sequences ---")
    for col in best_df.columns:
        if col.startswith("loop_") and col.endswith("_seq"):
            print(f"{col.replace('_seq', '')}: {top_design[col]}")
            
    print(f"\nBinding Probability: {top_design['binding_probability']:.2f}")
    print(f"pLDDT: {top_design['plddt']:.2f}")
    print(f"pTM:   {top_design['ptm']:.2f}")
    print(f"RMSD:  {top_design['overall_rmsd']:.2f} Å")
    
    best_cif_path = os.path.join(rf3_out_dir, top_design['file'].replace('.cif', '_refolded.cif'))
    
    if os.path.exists(best_cif_path):
        print(f"\nVisualizing: {os.path.basename(best_cif_path)}")
        b_min, b_max, invert = get_b_metrics(best_cif_path)
        
        if b_max - b_min < 0.0001: b_min, b_max, invert = 0.0, 1.0, False
        bound_min = b_max if invert else b_min
        bound_max = b_min if invert else b_max

        with open(best_cif_path, "r") as f:
            cif_data = f.read()
            
        view = py3Dmol.view(width=800, height=600)
        view.addModel(cif_data, 'cif')
        
        view.setStyle({'chain': fixed_chains[0]}, {'cartoon': {'color': 'gray', 'style': 'oval', 'thickness': 0.2}})
        view.setStyle({'chain': chain_to_design}, 
                     {'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'roygb', 'min': bound_min, 'max': bound_max}, 'thickness': 0.2}})
        
        view.zoomTo({'chain': chain_to_design})
        view.show()
    else:
        print(f"\nRefolded CIF not found for visualization: {best_cif_path}")
else:
    print("Summary metrics CSV not found.")


--- Top Design Sequences ---
loop_C: AGNSNNPRTPC
loop_EF: SQCDADGRCG

Binding Probability: 0.37
pLDDT: 0.66
pTM:   0.27
RMSD:  21.66 Å

Visualizing: design_2_mpnn0_refolded.cif


3Dmol.js failed to load for some reason. Please check your browser console for error messages.